# 第9周 · 周五：PSM 平衡性检验与敏感性分析——匹配得好不好，数据说了算\n\n> 难度：⭐⭐⭐⭐ | 定位：掌握PSM的匹配质量诊断和稳健性验证，能判断一个PSM分析是否可信\n> 前置依赖：周四PSM原理与实现 | 后续用途：W10-W12综合案例中的PSM诊断部分\n> 面试地位：能否做平衡性检验是区分"会用PSM"和"理解PSM"的关键\n\n---\n\n## 📘 区块1：核心概念与第一性原理\n\n### 1.1 为什么PSM需要"检验"——倾向得分模型是不完美的\n\nPSM的"成功"依赖于三个条件同时成立：\n1. 倾向得分模型正确指定（该包含的变量都包含了，关系形式正确）\n2. 共同支撑成立（两组的倾向得分有充分的重叠区域）\n3. 匹配后协变量平衡（匹配消除了两组在可观测变量上的差异）\n\n如果说周四的PSM是"跑模型"——今天的PSM检验就是"质检"——判断周四的结果能不能用。\n\n**三条检验线**：\n```\n检验1：平衡性检验(SMD)——匹配后两组在协变量上"长一样"了吗？\n检验2：共同支撑检查——两组倾向得分有足够重叠吗？\n检验3：敏感性分析——如果存在未观测混淆，结论有多脆弱？\n```\n\n### 1.2 SMD（标准化均值差）——匹配质量的"金标准"\n\nSMD衡量的是两组在某个变量上的均值差异有多大，标准化后可以在不同变量间比较。\n\n$$SMD = \\frac{\\bar{X}_{treated} - \\bar{X}_{control}}{\\sqrt{(s^2_{treated} + s^2_{control})/2}}$$\n\n**SMD的判断标准（行业共识）**：\n- SMD < 0.1：匹配良好 ✓\n- 0.1 ≤ SMD < 0.2：可接受 △\n- SMD ≥ 0.2：匹配不足 ✗（该变量仍有实质性差异）\n\n**为什么是0.1不是0.05**：0.1的标准来自Normand(2001)和Austin(2011)的系统性模拟研究。0.05太严（许多真实数据达不到），0.2太松（仍有明显偏差）。\n\n### 1.3 共同支撑——两组有没有"可比性"\n\n倾向得分的共同支撑（Common Support / Overlap）要求：对于每个处理组个体，能找到倾向得分"接近"的对照组个体。\n\n**如果共同支撑不成立**：处理组的人都集中在高分区域（ps>0.8），对照组的人都在低分区域（ps<0.2）——处理组的人"几乎一定会被处理"，对照组的人"几乎一定不会被处理"，两组本质上是不同的人群，无法比较。\n\n**检查方法**：\n1. 直方图/密度图叠加——直观\n2. 修剪规则：只保留ps在[min(ps_treated), max(ps_control)]和[min(ps_control), max(ps_treated)]交集内的样本\n\n### 1.4 敏感性分析——"多大的遗漏混淆才能推翻我的结论"\n\n这是PSM中最重要但也最常被跳过的步骤。\n\n**Rosenbaum敏感性分析(Bounds)**的核心问题：\n\"如果存在一个未观测的混淆变量，它要多强才能使我的结论从'显著'变成'不显著'？\"\n\n**关键参数Γ(Gamma)**：\n- Γ=1：没有未观测混淆（随机化）\n- Γ=2：两个特征完全相同的人，其中一个因为未观测因素被处理的概率是另一个的2倍\n- Γ越大→结论需要的"未观测混淆强度"越大→结论越稳健\n\n**解读示例**：如果当Γ=1.5时，p值已经>0.05了→说明只需要一个"使处理概率翻1.5倍"的未观测混淆就能推翻你的结论→结论脆弱。如果Γ=3时p值才>0.05→结论非常稳健。\n\n### 1.5 面试高频考点\n\n**面试官常问**：\n- "你做完了PSM，怎么判断匹配得好不好？"\n- "SMD<0.1就够了吗？还需要看什么？"\n- "共同支撑不成立怎么办？"\n- "你报告的PSM结果，我怎么知道不是因为遗漏了某个重要变量？"\n\n**回答要点**：\n- 看SMD(应<0.1)+倾向得分分布图(看重叠)+敏感性分析(多强混淆才能推翻结论)\n- SMD只检查了一阶矩(均值)，建议补看QQ图/方差比/高阶矩\n- 共同支撑不成立→修剪(truncation/trimming)→但要注意这改变了估计目标(从ATT变成了"重叠区域的ATT")\n- 这个问题只有敏感性分析能回答——Rosenbaum Bounds告诉你需要"多强"的遗漏变量

## 🧩 区块2：知识点详细拆解\n\n### 2.1 SMD诊断的四个维度\n\n**为什么重要**：只看SMD均值不够——还需要看分布的整体匹配程度。\n\n| 诊断维度 | 工具 | 标准 |\n|---------|------|------|\n| 均值平衡 | SMD | <0.1 |\n| 分布平衡 | QQ图 | 点接近45°对角线 |\n| 方差平衡 | 方差比 | 0.8~1.25 |\n| 整体平衡 | 倾向得分密度图 | 两条曲线基本重合 |\n\n**SMD的变体**：\n- 连续变量：用标准化的均值差\n- 二分类变量：用比例差除以合并标准差（也可以用Cramer's V）\n- 多分类变量：用多分类SMD（Mahalanobis距离的简化版）\n\n### 2.2 SMD的Love Plot——最直观的平衡性可视化\n\n**为什么重要**：Love Plot是PSM论文中必出现的一张图——一眼就能看出匹配前后所有变量的平衡性改善。\n\n**Love Plot的元素**：\n- 横轴：SMD值（绝对值）\n- 纵轴：各协变量（按匹配前的SMD排序）\n- 两种颜色/形状：匹配前(空心圆) vs 匹配后(实心圆)\n- 参考线：x=0.1（可接受阈值）和x=0（完美平衡）\n\n**好的Love Plot长什么样**：\n- 匹配前的点分散在右边（SMD大→不平衡）\n- 匹配后的点都聚集在0附近，全部在0.1参考线以内\n- → 一眼就能看到匹配效果\n\n### 2.3 QQ图诊断\n\n**为什么重要**：SMD只比较均值，QQ图比较整个分布——可能SMD<0.1但分布形状完全不同。\n\n**QQ图的构造**：\n对某个协变量，横轴=对照组的分位数，纵轴=处理组的分位数。匹配良好的话，点应该落在45°对角线上。\n\n**常见问题模式**：\n- 整体偏离对角线→均值有差异\n- S形偏离→方差不同\n- 尾部偏离→极值分布不同（可能是outlier导致的）\n\n### 2.4 Rosenbaum敏感性分析的数学直觉\n\n**为什么重要**：这是PSM中最被低估的步骤——但正是它让你的结论从"我匹配了"变成"匹配了且结论稳健"。\n\n**核心设定**：假设两个个体i和j，他们的可观测协变量完全相同（即倾向得分相同），但因一个未观测的混淆变量U，他们被处理的概率不同：\n\n$$\\frac{1}{Γ} \\leq \\frac{P(T_i=1|X_i)/P(T_i=0|X_i)}{P(T_j=1|X_j)/P(T_j=0|X_j)} \\leq Γ$$\n\n当Γ=1时→两个体被处理概率之比=1（没有未观测混淆）\n当Γ=2时→即使协变量完全相同，个体i被处理的优势(odds)可以是个体j的2倍\n\n**如何解读**：不断增加Γ，看Wilcoxon符号秩检验的p值上界何时超过0.05。如果Γ=1.2时p上界=0.06（>0.05）→结论极其脆弱（一个强度仅1.2的遗漏变量就能推翻它）。如果Γ=5时p上界仍<0.001→结论极其稳健。\n\n**实践标准**：在社会科学/医学研究中，Γ≥2而结论不变→通常被认为稳健。但这个标准依赖具体场景——如果已知有很强的未观测因素（如健康意识对医疗方案选择），需要更大的Γ。\n\n### 2.5 倾向得分修剪(Trimming)——处理共同支撑不足\n\n**为什么重要**：共同支撑不成立时，不是简单地"丢掉样本"——你需要理解修剪改变了你的估计目标。\n\n**三种修剪策略**：\n\n| 策略 | 做法 | 保留的样本 | 估计目标 |\n|------|------|-----------|----------|\n| 对称修剪 | 去掉ps<α和ps>1-α的样本 | 倾向得分"中间"的人 | 重叠区域的ATT |\n| 处理组修剪 | 只去掉ps超出对照组ps范围的处理组 | 大部分对照组+可匹配的处理组 | ATT(可匹配的处理组) |\n| 百分位修剪 | 去掉ps最低1%和最高1%的样本 | 去掉极端值后的样本 | 略有变化 |\n\n**关键认知**：修剪后估计的不是原始总体的ATT，而是"重叠区域的ATT"。这在报告中必须说明。\n\n### 2.6 伪R²和AUC——倾向得分模型的"性能"不应太好\n\n**为什么重要**：这是一个反直觉的点——倾向得分模型的C统计量(AUC)太高反而可能是坏消息！\n\n**两难困境**：\n- AUC太低(<0.6)→模型预测能力差→可能遗漏了重要混淆\n- AUC太高(>0.9)→处理分配近乎确定性→共同支撑可能很差(几乎完美分离了处理组和对照组)\n- AUC适中(0.65-0.85)→模型既捕捉了混淆，又保留了足够的重叠区域→理想状态\n\n**实际中的建议**：AUC>0.85→检查共同支撑（可能两组倾向得分分布几乎没有重叠），AUC<0.6→检查是否有重要混淆被遗漏。\n\n### 2.7 多重假设检验下的SMD——Bonferroni还是FDR？\n\n**为什么重要**：当你有30个协变量时，按α=0.05的标准，你期望有1.5个协变量"偶然地"看起来不平衡。\n\n**处理方案**：\n- 不强调每个变量的显著性检验（不要为每个变量跑t检验看p值）——SMD是一个效应量指标，不依赖样本量\n- 如果要检验，用FDR(Benjamini-Hochberg)校正p值\n- 最好的做法：报告所有变量的SMD，用Love Plot可视化（不依赖p值）

## 💻 区块3：动手练习\n\n### 步骤0：环境准备

In [ ]:
import numpy as np; import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from scipy import stats
import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print('='*60)
print('PSM平衡性检验与敏感性分析——动手练习')
print('='*60)

### 步骤1：基础练习——SMD计算与Love Plot

1.1 计算匹配前后各协变量的SMD
1.2 画Love Plot展示平衡性改善

In [ ]:
# 步骤1.1：模拟数据并进行PSM匹配
print('【步骤1.1】准备PSM数据并匹配')
print('-'*50)

n = 2000
# 多个协变量
X1 = np.random.normal(0, 1, n)  # 能力
X2 = np.random.normal(0, 1, n)  # 家境
X3 = np.random.normal(0, 1, n)  # 年龄(标准化)
X4 = np.random.normal(0, 1, n)  # 经验
X5 = np.random.binomial(1, 0.5, n)  # 性别
X6 = np.random.poisson(5, n)  # 历史购买次数

# 处理分配：跟多个协变量相关（制造不平衡）
prob_treat = 1 / (1 + np.exp(-(-0.5 + 0.8*X1 + 0.6*X2 + 0.3*X3 + 0.4*X4 + 0.2*X5 + 0.1*X6)))
treated = np.random.binomial(1, prob_treat)

# 结果
Y = 30 + 5*X1 + 3*X2 + 2*X3 + 1.5*X4 + 1*X5 + 0.5*X6 + 8*treated + np.random.normal(0, 5, n)

df = pd.DataFrame({'X1': X1, 'X2': X2, 'X3': X3, 'X4': X4, 'X5': X5, 'X6': X6, 
                   'treated': treated, 'Y': Y})

covariates = ['X1', 'X2', 'X3', 'X4', 'X5', 'X6']

# 计算匹配前SMD
def calc_smd(df, treated_col, var):
    t_mean = df[df[treated_col]==1][var].mean()
    c_mean = df[df[treated_col]==0][var].mean()
    t_var = df[df[treated_col]==1][var].var()
    c_var = df[df[treated_col]==0][var].var()
    pooled_std = np.sqrt((t_var + c_var) / 2)
    return (t_mean - c_mean) / pooled_std if pooled_std > 0 else 0

smd_before = {v: calc_smd(df, 'treated', v) for v in covariates}
print('匹配前SMD:')
for v, s in smd_before.items():
    flag = '⚠ >0.2!' if abs(s) > 0.2 else ('△ 0.1-0.2' if abs(s) > 0.1 else '✓')
    print(f'  {v}: SMD={s:+.3f} {flag}')

# PSM匹配
lr = LogisticRegression(max_iter=1000)
lr.fit(df[covariates], df['treated'])
df['ps'] = lr.predict_proba(df[covariates])[:, 1]

treated_df = df[df.treated==1].copy()
control_df = df[df.treated==0].copy()
matched_control_idx = []
for _, row in treated_df.iterrows():
    idx = (control_df['ps'] - row['ps']).abs().idxmin()
    matched_control_idx.append(idx)
    control_df = control_df.drop(idx)

matched_df = df.loc[matched_control_idx]
matched_combined = pd.concat([treated_df, matched_df])

# 计算匹配后SMD
smd_after = {v: calc_smd(matched_combined, 'treated', v) for v in covariates}
print('\n匹配后SMD:')
for v, s in smd_after.items():
    flag = '⚠' if abs(s) > 0.2 else ('△' if abs(s) > 0.1 else '✓ <0.1')
    print(f'  {v}: SMD={s:+.3f} {flag}')

In [ ]:
# 步骤1.2：Love Plot
print('\n【步骤1.2】Love Plot——匹配前后SMD对比')
print('-'*50)

fig, ax = plt.subplots(figsize=(10, 6))

# 按匹配前的SMD绝对值排序（用于纵轴顺序）
var_order = sorted(covariates, key=lambda v: abs(smd_before[v]), reverse=True)

y_pos = range(len(covariates))
for i, var in enumerate(var_order):
    ax.scatter(smd_before[var], i, color='red', s=100, marker='o', facecolors='none', label='匹配前' if i==0 else '')
    ax.scatter(smd_after[var], i, color='blue', s=100, marker='o', label='匹配后' if i==0 else '')
    ax.plot([smd_before[var], smd_after[var]], [i, i], color='gray', linewidth=1, alpha=0.5)

ax.axvline(x=0.1, color='green', linestyle='--', linewidth=1.5, label='SMD=0.1(良好)')
ax.axvline(x=-0.1, color='green', linestyle='--', linewidth=1.5)
ax.axvline(x=0.2, color='orange', linestyle=':', linewidth=1, label='SMD=0.2(可接受)')
ax.axvline(x=-0.2, color='orange', linestyle=':', linewidth=1)
ax.axvline(x=0, color='black', linewidth=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(var_order)
ax.set_xlabel('SMD(标准化均值差)')
ax.set_title('Love Plot: 匹配前后协变量平衡性对比')
ax.legend(loc='lower right')
plt.tight_layout(); plt.show()

print('\n解读：蓝点比红点更靠近0→匹配改善了平衡性')
print('所有蓝点在±0.1以内→匹配良好 ✓')
print('如果有蓝点在±0.1以外→该变量匹配不足→考虑加入二次项或交互项重新估计ps')

### 步骤2：进阶练习——共同支撑与敏感性分析

2.1 共同支撑区域检查
2.2 Rosenbaum敏感性分析（简化版）

In [ ]:
# 步骤2.1：共同支撑检查
print('\n【步骤2.1】共同支撑区域检查')
print('-'*50)

ps_treated = df[df.treated==1]['ps']
ps_control = df[df.treated==0]['ps']

overlap_min = max(ps_treated.min(), ps_control.min())
overlap_max = min(ps_treated.max(), ps_control.max())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：直方图+共同支撑区域
axes[0].hist(ps_treated, bins=40, alpha=0.5, color='red', label='处理组', density=True)
axes[0].hist(ps_control, bins=40, alpha=0.5, color='blue', label='对照组', density=True)
axes[0].axvspan(overlap_min, overlap_max, alpha=0.15, color='green', label='共同支撑区域')
axes[0].set_xlabel('倾向得分'); axes[0].set_title('倾向得分分布与共同支撑')
axes[0].legend()

# 右图：QQ图（检查倾向得分的分布匹配）
from scipy.stats import probplot
# 使用分位数-分位数比较
q_treated = np.quantile(ps_treated, np.linspace(0.01, 0.99, 50))
q_matched = np.quantile(matched_df['ps'], np.linspace(0.01, 0.99, 50))
axes[1].scatter(q_matched, q_treated, alpha=0.6)
axes[1].plot([0, 1], [0, 1], 'r--', linewidth=1, label='45°线(完美匹配)')
axes[1].set_xlabel('匹配后对照组ps分位数'); axes[1].set_ylabel('处理组ps分位数')
axes[1].set_title('QQ图：倾向得分分布对比')
axes[1].legend()
plt.tight_layout(); plt.show()

# 不在共同支撑区域的样本比例
off_support = ((df['ps'] < overlap_min) | (df['ps'] > overlap_max)).sum()
print(f'共同支撑区域: [{overlap_min:.3f}, {overlap_max:.3f}]')
print(f'不在共同支撑区域的样本: {off_support}人 ({off_support/n*100:.1f}%)')

# 重叠不足的处理组个体（在对照组找不到接近匹配的）
caliper = 0.2 * df['ps'].std()
treated_ps_vals = treated_df['ps'].values
control_ps_vals = df[df.treated==0]['ps'].values
min_distances = np.min(np.abs(treated_ps_vals[:, None] - control_ps_vals[None, :]), axis=1)
unmatchable = (min_distances > caliper).sum()
print(f'在caliper={caliper:.4f}下找不到匹配的处理组: {unmatchable}人 ({unmatchable/len(treated_ps_vals)*100:.1f}%)')
if unmatchable / len(treated_ps_vals) > 0.1:
    print('  → 超过10%处理组找不到匹配 ⚠ 共同支撑不足，建议修剪或放宽caliper')

In [ ]:
# 步骤2.2：Rosenbaum敏感性分析（简化实现）
print('\n【步骤2.2】Rosenbaum敏感性分析')
print('-'*50)

# Rosenbaum bounds 的简化版实现
# 原理：对每个匹配对，生成"未观测混淆使处理组更优"和"使对照组更优"两种极端配对的p值范围

from scipy.stats import wilcoxon, norm

# 收集匹配对的结果差值（处理组 - 匹配对照组）
matched_outcomes = matched_df['Y'].values
treated_outcomes = treated_df['Y'].values
pair_diffs = treated_outcomes - matched_outcomes  # 每对的处理效应

# Wilcoxon符号秩检验（基准：无未观测混淆，Γ=1）
stat, p_base = wilcoxon(pair_diffs, alternative='greater')
print(f'Γ=1 (无未观测混淆) 时p值: {p_base:.6f}')

# 对不同的Γ值，计算p值的上界和下界
gammas = [1.0, 1.2, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]
print('\nΓ (Gamma)  | p值上界   | 结论')
print('-'*45)

for gamma in gammas:
    # 简化计算：在给定Γ下，Wilcoxon统计量的期望和方差会偏移
    # 这是Rosenbaum bounds的精简版——完整版需要更复杂的计算
    n_pairs = len(pair_diffs)
    
    # Rosenbaum: 调整后的"优势"概率
    p_plus = gamma / (1 + gamma)  # 处理组"更高"的概率上界
    
    # 在p_plus下，Wilcoxon统计量的期望和方差
    T_exp = n_pairs * (n_pairs + 1) / 2 * p_plus
    T_var = n_pairs * (n_pairs + 1) * (2*n_pairs + 1) / 6 * p_plus * (1 - p_plus)
    
    # 用正态近似计算p值上界
    T_obs = sum(np.where(pair_diffs > 0)[0] + 1)  # 正的秩和
    z = (T_obs - T_exp) / np.sqrt(T_var)
    p_upper = 1 - norm.cdf(z)
    
    conclusion = '✓ 显著' if p_upper < 0.05 else '✗ 不显著'
    print(f'  {gamma:.1f}        | {p_upper:.6f}  | {conclusion}')

print('\n解读：')
print('  Γ=1: 没有未观测混淆时的p值')
print('  Γ越大: 需要的未观测混淆越强')
print('  找到p值首次>0.05的Γ→这就是结论"临界脆弱点"')
print('  如果Γ>2时结论仍然显著→结论被认为是稳健的（Cohen标准）')

### 步骤3：挑战练习——综合诊断报告

3.1 生成一份完整的PSM诊断报告
3.2 包含SMD、Love Plot、共同支撑、敏感性分析

In [ ]:
# 步骤3：综合诊断——生成PSM完整诊断报告
print('\n【步骤3】PSM综合诊断报告')
print('='*60)

# 1. AUC诊断
from sklearn.metrics import roc_auc_score
auc = roc_auc_score(df['treated'], df['ps'])
print(f'\n【倾向得分模型诊断】')
print(f'  AUC = {auc:.3f}')
if auc < 0.6:
    print('  → AUC偏低，可能遗漏重要混淆变量 ⚠')
elif auc > 0.85:
    print('  → AUC偏高，两组可能难以匹配(共同支撑差) ⚠')
else:
    print('  → AUC适中，模型捕捉了混淆且保留了重叠 ✓')

# 2. 共同支撑总结
overlap_pct = 1 - off_support / n
print(f'\n【共同支撑诊断】')
print(f'  共同支撑区域: [{overlap_min:.3f}, {overlap_max:.3f}]')
print(f'  在共同支撑内的样本比例: {overlap_pct*100:.1f}%')
if overlap_pct > 0.9:
    print('  → 共同支撑良好 ✓')
else:
    print('  → 超过10%样本不在共同支撑区域 ⚠ 建议修剪')

# 3. SMD总结
print(f'\n【平衡性诊断】')
max_smd_before = max(abs(s) for s in smd_before.values())
max_smd_after = max(abs(s) for s in smd_after.values())
mean_smd_before = np.mean([abs(s) for s in smd_before.values()])
mean_smd_after = np.mean([abs(s) for s in smd_after.values()])
print(f'  匹配前: 最大SMD={max_smd_before:.3f}, 平均|SMD|={mean_smd_before:.3f}')
print(f'  匹配后: 最大SMD={max_smd_after:.3f}, 平均|SMD|={mean_smd_after:.3f}')
if max_smd_after < 0.1:
    print('  → 所有变量SMD<0.1，平衡性良好 ✓')
elif max_smd_after < 0.2:
    print('  → 有变量0.1≤SMD<0.2，可接受但需注意 △')
else:
    print('  → 有变量SMD≥0.2，匹配不足 ✗ 需要重新建模')

# 4. ATT估计
att_estimate = treated_outcomes.mean() - matched_outcomes.mean()
att_se = np.std(treated_outcomes - matched_outcomes) / np.sqrt(len(pair_diffs))
att_ci = (att_estimate - 1.96*att_se, att_estimate + 1.96*att_se)
print(f'\n【效应估计】')
print(f'  ATT = {att_estimate:.2f} (95% CI: [{att_ci[0]:.2f}, {att_ci[1]:.2f}])')
print(f'  真实效应 = 8.0')
print(f'  偏误 = {att_estimate - 8:.2f}')

# 5. 总体诊断结论
print(f'\n【总体诊断】')
checks_passed = sum([
    0.6 <= auc <= 0.85,
    overlap_pct > 0.9,
    max_smd_after < 0.2,
    att_ci[0] > 0 if att_estimate > 0 else True
])
print(f'  通过检验: {checks_passed}/4')
if checks_passed == 4:
    print('  → PSM分析通过全部诊断 ✓ 结果可信')
elif checks_passed >= 2:
    print('  → PSM分析部分通过诊断 △ 建议在报告中说明局限性')
else:
    print('  → PSM分析未通过关键诊断 ✗ 需要重新设计')

## 📊 区块4：业务场景应用

### 场景1：一篇PSM分析被挑战——你怎么回应

**背景**：你给业务方做了一份"VIP会员的增量消费"PSM分析，结论是VIP带来约200元/月的增量。合作方提出了以下质疑，你需要逐一回应：

**质疑1**："你怎么知道匹配后两组真的'像'了？"
- 回应：展示Love Plot——所有协变量的匹配后SMD都<0.1。附上共同支撑图——两组倾向得分有充分的重叠。

**质疑2**："也许还有你没测量到的变量（如品牌忠诚度）在影响'是否开VIP'和'消费金额'——你的结论可能是假的。"
- 回应：这正是PSM的局限性。我做了Rosenbaum敏感性分析——结论在Γ=2.5时仍然显著，意味着需要一个能"使人开VIP的优势增加2.5倍的遗漏变量"才能推翻结论。考虑到我能观测到的最大混淆变量(历史消费)的效应约在Γ=1.8左右，2.5意味着这个遗漏变量要比历史消费强得多——不太可能。

**质疑3**："你丢掉了15%的样本（不在共同支撑区域）——结论还适用于全体VIP用户吗？"
- 回应：好问题。丢掉的15%主要是"极高倾向得分"的用户——他们几乎100%会开VIP（所以找不到可比的人）。我的结论只适用于"倾向得分在重叠区域"的VIP用户（约占VIP的85%）。对于那些"铁定会开VIP"的用户，我没有可行的对照组来估计因果效应。

### 场景2：一个PSM分析出问题了——诊断与修复

**现象**：某分析师做了PSM来评估"参加培训项目"对"工资"的影响。检查发现：
- AUC=0.92（太高！→共同支撑差）
- 匹配后SMD：教育年限0.25、工作经验0.18（多个变量不通过）
- 共同支撑区域只有60%样本

**诊断**：倾向得分模型"太好"了——几乎完美预测了谁参加培训（因为参加培训的人特征太鲜明了：都是高学历低经验的年轻人）。但正是这种"太好"导致了共同支撑差——找不到"特征相似但没参加培训"的对照组。

**修复方案**：
1. 修剪样本——只保留ps在[0.1, 0.9]的人（但这样会损失很多处理组样本）
2. 简化倾向得分模型——只用最关键的几个混淆变量，接受AUC低一些但共同支撑更好
3. 换方法——考虑IV（如用"距离最近培训中心"作为工具变量）或RDD（如培训有个分数线）
4. 诚实地说"数据不支持用PSM可靠地估计这个因果效应"

**教训**：不是所有问题都适合用PSM。如果处理分配几乎是"确定性的"（基于特征的强选择），PSM无法构建可信的反事实。

### 场景3：面试中的PSM诊断题

**面试题**："我给你看一个PSM分析的Love Plot——匹配前SMD都在0.5-1.0之间，匹配后都在0.02-0.08之间，全部<0.1。AUC=0.72，共同支撑89%。这个PSM做得怎么样？"

**你的回答**：

1. "首先，这个PSM的匹配质量很好——所有变量SMD<0.1，AUC适中（0.72），共同支撑近90%。这些指标表明，在可观测变量上，匹配是成功的。"

2. "但有几个问题我需要追问：
   - 倾向得分模型包含了哪些变量？是如何选择的？有没有遗漏重要的混淆变量？
   - 有没有做Rosenbaum敏感性分析？Love Plot只证明可观测变量平衡了，但未观测混淆的威胁仍然存在。
   - 11%样本不在共同支撑区域——这些人是谁？结论能推广到他们吗？
   - 匹配方法是什么？报告的结果对匹配方法选择是否敏感？"

3. "总的来说，就展示的指标而言，这个PSM的'技术执行'很好。但'因果推断的有效性'取决于一个无法从这些图中回答的问题——是否遗漏了重要的混淆变量。我建议补充敏感性分析。

## ✅ 区块5：分级检验清单

### 🟢 初级检验（必须能独立完成的）
- [ ] 能计算并解读SMD（标准：<0.1良好, 0.1-0.2可接受, >0.2不足）
- [ ] 能画出并解读Love Plot（匹配前后SMD对比）
- [ ] 能画出并判断倾向得分分布的直方图（看共同支撑）
- [ ] 能计算倾向得分模型的AUC并给出诊断

**答案提示**：
- SMD = (均值差) / 合并标准差，不受样本量影响（是效应量）
- Love Plot: 横轴SMD，纵轴变量，两种形状代表匹配前后
- 共同支撑：两组ps直方图的重叠区域；重叠充分→可匹配→有可比性
- AUC 0.65-0.85理想；太高→共同支撑差；太低→遗漏混淆

### 🟡 中级检验（能解释给同事听的原理）
- [ ] 能解释为什么SMD比p值更适合做平衡性诊断
- [ ] 能说明Rosenbaum敏感性分析回答什么问题
- [ ] 能判断共同支撑不足时的处理策略
- [ ] 能解释为什么AUC太高反而是坏消息

**答案提示**：
- SMD是效应量指标，不依赖样本量；p值在大样本下总是"显著"（但SMD未必大）
- Rosenbaum分析回答："要多大的未观测混淆才能推翻结论"
- 共同支撑不足→修剪/简化模型/换方法/诚实说"无法估计"
- AUC太高→两组近乎完美分离→几乎找不到ps相似的匹配→共同支撑差

### 🔴 高级检验（能举一反三的迁移能力）
- [ ] 能设计一个包含所有诊断维度的PSM分析方案
- [ ] 能根据诊断结果判断是否需要"退回重新建模"
- [ ] 能解释为什么PSM的结论应该用"多个匹配方法"做敏感性检验
- [ ] 能判断一个PSM分析的"致命缺陷"在哪里

**答案提示**：
- 完整方案：AUC→共同支撑→SMD(Love Plot)→QQ图→Rosenbaum→多方法(1:1/1:3/卡尺/核匹配)稳健性检验
- 退回重建的信号：多个变量SMD>0.2/共同支撑<70%/AUC<0.55或>0.95
- 不同匹配方法给出不同结果→结论不稳健→报告这个不稳健性本身也是发现
- PSM的致命缺陷永远是"未观测混淆"→如果领域知识告诉你存在强未观测混淆，PSM不可行

## 🔗 区块6：跨模块连接

| 连接方向 | 关联内容 | 为什么有关 |
|---------|---------|----------|
| **← 前置依赖** | 周四PSM原理与实现 | 先匹配，再诊断——这是标准流程 |
| **← 前置依赖** | 效应量(Cohen's d) | SMD本质上是Cohen's d的变体 |
| **→ 后续用途** | W10-W12综合案例中的PSM诊断和报告 | 综合案例的PSM部分必须包含诊断 |
| **业务连接** | 所有PSM分析的"质检"环节 | 任何PSM分析在出结论前都必须过这一关 |
| **学术连接** | 论文中的PSM必须报告平衡性检验和稳健性 | 期刊审稿人会首先检查SMD和Love Plot |

### 横向连接：与其他分析方法的关联

**与DID检验的连接**：
- PSM的平衡性检验 ≈ DID的平行趋势检验——都是检验"核心假设是否成立"
- PSM的敏感性分析 ≈ DID的灵敏度分析(Rambachan-Roth)——都是问"假设偏了结论还稳吗"

**与AB实验的连接**：
- 实验前用SMD检查随机化是否成功（baseline balance check）
- 实验中用SMD监控MRLC(Minimum Relevant Lift Check)——与PSM共享诊断工具

**与可视化的连接**：
- Love Plot是因果推断领域最具辨识度的可视化之一
- 好的可视化让非技术stakeholder也能判断分析可信度

## 📝 区块7：当日小结

> **PSM做完不等于做对了——平衡性检验是PSM的"安检"。核心诊断工具：SMD（<0.1匹配良好）、Love Plot（匹配前后SMD对比）、共同支撑图（两组ps分布重叠）、Rosenbaum敏感性分析（多强的遗漏混淆能推翻结论）。记住：SMD比p值更适合做平衡性诊断（SMD是效应量，不被样本量影响）。AUC太高（>0.85）反而是坏消息——意味着两组在ps上几乎完全分离，共同支撑差，匹配不可行。倾向得分模型不是"越准越好"——需要在"捕捉混淆"和"保留重叠"之间取得平衡。Rosenbaum敏感性分析是PSM的"最后防线"——它告诉你结论对未观测混淆有多脆弱。一个好的PSM结论应该是："在可观测变量条件下，VIP的效果是X。Rosenbaum分析表明，需要一个强度至少Y的遗漏混淆才能推翻这一结论。"**

**三条金句带回家**：
1. SMD<0.1是匹配成功的"最低门槛"——不如<0.1，回去重新调倾向得分模型
2. Love Plot是PSM最重要的图——一张图同时展示匹配前后所有变量的平衡性变化
3. 敏感性分析是PSM的"弱点声明"——诚实地告诉你"结论有多容易被推翻"

### 🔖 课后拓展

1. **练习**：在周四生成的PSM数据上，做完整的平衡性检验+敏感性分析，写一份不超过1页的诊断报告

2. **思考**：如果Love Plot显示"匹配后所有SMD<0.1"，但Rosenbaum分析显示"Γ=1.3时p值就>0.05了"，这说明什么？（提示：可观测的匹配好了，但处理效应本身太弱/样本量太小→即使没有混淆也难以检测到）

3. **预习**：明天要学的IV和RDD——当PSM不适用时的替代方案

4. **面试准备**：
   - "你怎么判断一个PSM分析做得好不好？"
   - "SMD<0.1就够了吗？还需要看什么？"
   - "如果有人说'我做了PSM并证明了因果效应'，你会怎么质疑？"

---

*03analyst_thinking · Week 9 Day 5 · 2026-05-21*